In [2]:
import json
import os

# ==========================================
# 1. FOOTER FLATTENER
# ==========================================
def flatten_footer_to_string(data):
    if isinstance(data, dict):
        parts = []
        for k, v in data.items():
            clean_key = str(k).replace('_', ' ').title() 
            if isinstance(v, (dict, list)):
                parts.append(f"{clean_key}: {flatten_footer_to_string(v)}")
            else:
                parts.append(f"{clean_key}: {v}")
        return " | ".join(parts)
    elif isinstance(data, list):
        return ", ".join(flatten_footer_to_string(i) for i in data)
    else:
        return str(data).strip()

# ==========================================
# 2. SPEC PRUNER (Removes Data Leakage)
# ==========================================
def prune_chart_spec(spec):
    if not isinstance(spec, dict): 
        return spec
    
    # Keep only the essential global metadata
    clean_spec = {
        "title": spec.get("title", ""),
        "topo": spec.get("topo", {}),
        "axes": spec.get("axes", []),
        "footer": spec.get("footer", {})
    }
    
    # Rebuild the series array with ONLY raw data and visuals
    clean_series = []
    for s in spec.get("ser", []):
        if not isinstance(s, dict): continue
        clean_series.append({
            "name": s.get("name", "Unknown"),
            "color": s.get("color", "None"),
            "y_ref": s.get("y_ref", "y_axis"),
            "data": s.get("data", [])
        })
    
    clean_spec["ser"] = clean_series
    return {k: v for k, v in clean_spec.items() if v}

# ==========================================
# 3. UNIVERSAL MARKDOWN PARSER
# ==========================================
def refine_md_parser(raw_spec_json):
    if not raw_spec_json: return "No Spec Available"
        
    if isinstance(raw_spec_json, str):
        try: spec = json.loads(raw_spec_json)
        except: return "Invalid JSON Spec"
    elif isinstance(raw_spec_json, dict):
        spec = raw_spec_json
    else:
        return "Unknown Spec Format"

    # 🔥 Pass through the pruner to prevent LLM cheating
    spec = prune_chart_spec(spec)
    md_lines = []
    
    # --- Metadata Section ---
    title = spec.get("title", "")
    if title: md_lines.append(f"**Title:** {title}")
        
    topo = spec.get("topo", {})
    t_type_str = ""
    if isinstance(topo, dict):
        t_type = topo.get("type", "")
        t_sub = topo.get("sub", "")
        if isinstance(t_type, list): t_type = "+".join(t_type)
        t_type_str = str(t_type).lower()
        
        chart_type_str = f"{str(t_sub).capitalize()} {str(t_type).capitalize()} Chart" if t_sub else f"{str(t_type).capitalize()} Chart"
        md_lines.append(f"**Chart Type:** {chart_type_str.strip()}")
    
    axes = spec.get("axes", [])
    ax_strs = []
    if isinstance(axes, list):
        for ax in axes:
            if isinstance(ax, dict):
                name, scl, ax_title = ax.get("name", ""), ax.get("scl", ""), ax.get("lbl", "")
                ax_info = []
                if ax_title: ax_info.append(f"'{ax_title}'")
                if scl: ax_info.append(f"{scl} scale")
                    
                if "x_" in name.lower(): ax_strs.append(f"{name.lower()} ({', '.join(ax_info)})" if ax_info else "X-Axis")
                elif "y_" in name.lower(): ax_strs.append(f"{name.lower()} ({', '.join(ax_info)})" if ax_info else "Y-Axis")
            elif isinstance(ax, str): ax_strs.append(ax)
                
    if ax_strs: md_lines.append(f"**Axes:** {', '.join(ax_strs)}")
        
    md_lines.append("\n### Data & Visuals Table")
    
    # --- Series Validation ---
    series_list = spec.get("ser", [])
    valid_series = [s for s in series_list if isinstance(s, dict)] if isinstance(series_list, list) else []
    
    if not valid_series:
        md_lines.append("No valid structured data series found.")
        return "\n".join(md_lines)
    
    # ==========================================
    # 4. CHART-TYPE DATA ROUTER
    # ==========================================
    is_pie = "pie" in t_type_str
    is_box = "box" in t_type_str

    if is_pie:
        # [PIE CHARTS] -> 1D Value Mapping
        md_lines.append("| Category (Series) | Color | Value |")
        md_lines.append("| :--- | :--- | :--- |")
        for ser in valid_series:
            name, color = ser.get("name", "Unknown"), ser.get("color", "None")
            data = ser.get("data", [])
            val = data[0][0] if data and isinstance(data, list) and len(data)>0 and isinstance(data[0], list) and len(data[0])>0 else "-"
            if isinstance(val, float): val = round(val, 4)
            md_lines.append(f"| {name} | {color} | {val} |")
            
    elif is_box:
        # [BOX PLOTS] -> 5-Number Summary
        md_lines.append("| X-Value (Category) | Series | Min | Q1 | Median | Q3 | Max |")
        md_lines.append("| :--- | :--- | :--- | :--- | :--- | :--- | :--- |")
        for ser in valid_series:
            header = f"{ser.get('name', 'Unknown')} ({ser.get('color', 'None')})"
            for pt in ser.get("data", []):
                if isinstance(pt, list) and len(pt) >= 6:
                    x_cat = pt[0]
                    v = [round(x, 4) if isinstance(x, float) else x for x in pt[1:6]]
                    md_lines.append(f"| {x_cat} | {header} | {v[0]} | {v[1]} | {v[2]} | {v[3]} | {v[4]} |")
                    
    else:
        # [BAR, LINE, AREA, SCATTER, HISTOGRAM] -> Universal Pivot Table
        x_values = []
        data_dict = {} 
        series_headers = []
        
        for ser in valid_series:
            name, color = ser.get("name", "Unknown"), ser.get("color", "None")
            header = f"{name} (Color: {color})"
            if header not in series_headers: series_headers.append(header)
                
            for pt in ser.get("data", []):
                if isinstance(pt, list) and len(pt) >= 2:
                    
                    # Route Histogram Bins [start, end, density]
                    if len(pt) >= 3 and "histogram" in t_type_str:
                        xs, xe, y_val = pt[0], pt[1], pt[2]
                        if isinstance(xs, float): xs = round(xs, 4)
                        if isinstance(xe, float): xe = round(xe, 4)
                        x_val = f"{xs} to {xe}"
                    else:
                        x_val, y_val = pt[0], pt[1]
                        
                    if isinstance(y_val, float): y_val = round(y_val, 4)
                    
                    if x_val not in data_dict:
                        data_dict[x_val] = {}
                        x_values.append(x_val)
                    if header not in data_dict[x_val]:
                        data_dict[x_val][header] = []
                    
                    # Prevent artifact duplication
                    if not data_dict[x_val][header] or data_dict[x_val][header][-1] != y_val:
                        data_dict[x_val][header].append(y_val)
                        
        header_row = f"| X-Value | " + " | ".join(series_headers) + " |"
        sep_row = "| :--- | " + " | ".join([":---"] * len(series_headers)) + " |"
        md_lines.append(header_row)
        md_lines.append(sep_row)
        
        for x_val in x_values:
            row = [str(x_val)]
            for sh in series_headers:
                vals = data_dict[x_val].get(sh, ["-"])
                row.append(", ".join(map(str, vals)))
            md_lines.append("| " + " | ".join(row) + " |")

    # --- Footer Section ---
    footer = spec.get("footer")
    if footer:
        md_lines.append("\n### Footer Metadata")
        md_lines.append(flatten_footer_to_string(footer))

    return "\n".join(md_lines)

# ==========================================
# 5. EXECUTION LOOP
# ==========================================
FILES_TO_PROCESS = ["train_w_spec.json","validation_w_spec.json","test_1_w_spec.json","test_2_w_spec.json"] # Replace with your train/test list

for filepath in FILES_TO_PROCESS:
    if not os.path.exists(filepath):
        print(f"Skipping {filepath}, file not found.")
        continue
        
    print(f"Parsing and Pruning {filepath}...")
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    for item in data:
        item["enriched_markdown"] = refine_md_parser(item.get("extended_spec"))
        
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)
        
    print(f"✅ Successfully extracted clean Markdown to {len(data)} items in {filepath}.")

Parsing and Pruning train_w_spec.json...
✅ Successfully extracted clean Markdown to 7607 items in train_w_spec.json.
Parsing and Pruning validation_w_spec.json...
✅ Successfully extracted clean Markdown to 953 items in validation_w_spec.json.
Parsing and Pruning test_1_w_spec.json...
✅ Successfully extracted clean Markdown to 939 items in test_1_w_spec.json.
Parsing and Pruning test_2_w_spec.json...
✅ Successfully extracted clean Markdown to 981 items in test_2_w_spec.json.


In [5]:
import json
import random

FILE_TO_CHECK = "test_2_w_spec.json"

def sanity_check_markdown(filepath):
    print(f"--- Sanity Check: {filepath} ---")
    
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    total_items = len(data)
    has_md_count = 0
    failed_parse_count = 0
    
    # Track items that successfully generated a markdown table
    successful_items = []
    
    for item in data:
        md = item.get("enriched_markdown", "")
        if md:
            has_md_count += 1
            # Check if it hit one of our error fallbacks
            if md in ["No Spec Available", "Invalid JSON Spec", "Unknown Spec Format", "No valid structured data series found.", "No data series found."]:
                failed_parse_count += 1
            else:
                successful_items.append(item)
                
    print(f"Total charts in file: {total_items}")
    print(f"Charts with 'enriched_markdown' field: {has_md_count}")
    print(f"Charts that failed/skipped parsing: {failed_parse_count}")
    print(f"Successfully parsed into Markdown: {len(successful_items)}")
    print("-" * 40)
    
    # Print 3 random successful examples to visually verify the tables
    if successful_items:
        print("\n👀 VISUAL INSPECTION (3 Random Charts) 👀\n")
        samples = random.sample(successful_items, min(3, len(successful_items)))
        
        for i, sample in enumerate(samples):
            print(f"========== SAMPLE {i+1} ==========")
            print(f"CLAIM: {sample.get('claim', 'No claim')}")
            print(f"GROUND TRUTH: {sample.get('label', 'Unknown')}\n")
            print("--- GENERATED MARKDOWN ---")
            print(sample["enriched_markdown"])
            print("\n" + "="*30 + "\n")

sanity_check_markdown(FILE_TO_CHECK)

--- Sanity Check: test_2_w_spec.json ---
Total charts in file: 981
Charts with 'enriched_markdown' field: 981
Charts that failed/skipped parsing: 0
Successfully parsed into Markdown: 981
----------------------------------------

👀 VISUAL INSPECTION (3 Random Charts) 👀

========== SAMPLE 1 ==========
CLAIM: Income tax is the largest source of Ohio state revenue.
GROUND TRUTH: True

--- GENERATED MARKDOWN ---
**Title:** Composition of State and Local Government Tax Revenue for Ohio, 2007
**Chart Type:** Standard Pie Chart

### Data & Visuals Table
| Category (Series) | Color | Value |
| :--- | :--- | :--- |
| Income Taxes | #8c564b | 0.32 |
| Property Taxes | #c44e52 | 0.29 |
| Sales Taxes | #4c72b0 | 0.31 |
| Other | #55a868 | 0.07 |


========== SAMPLE 2 ==========
CLAIM: The number of files with no language templates is equal to the number of files with four or more language templates.
GROUND TRUTH: False

--- GENERATED MARKDOWN ---
**Title:** Number of files with N language templates